#### 0. Realizar a limpeza dos diretórios para evitar falhas de execução

In [ ]:
%%bash
rm -rf ../../data/1-sot/*

#### 1. Realizar a importação dos módulos

In [ ]:
import duckdb
import matplotlib.pyplot as matplotlib
import os

#### 2. Realizar a captura dos arquivos da camada SOR

In [ ]:
sor_files_list = os.listdir("../../data/0-sor/")
sor_files_list

#### 3. Realizar a gravação de uma amostra de 5% dos dados de cada arquivo parquet da camada SOR

3.1. Além disso, são removidos registros que possuam ao menos um atributo sem informação (vazio ou nulo).

In [ ]:
con = duckdb.connect()

for sor_file in sor_files_list:
    
    input_path = f"../../data/0-sor/{sor_file}"
    filename = os.path.basename(sor_file)
    output_path = f"../../data/1-sot/sample-{filename}"
    
    query = f"""
    COPY 
    (
        SELECT 
            *
        FROM 
            read_parquet("{input_path}", union_by_name=true)
        WHERE 
            hash(request_datetime) % 100 < 5
    )
    TO "{output_path}" (FORMAT "parquet", COMPRESSION "snappy");
    """
    
    con.execute(query)

con.close()

#### 4. Realizar a gravação de um arquivo unificado considerando todas as amostras geradas

In [ ]:
con = duckdb.connect()

input_path = "../../data/1-sot/*.parquet"
output_path = "../../data/1-sot/unified-tb-sor-fhvhv-trip-data.parquet"

query = f"""
COPY 
(
    SELECT 
        *
    FROM 
        read_parquet("{input_path}")
)
TO "{output_path}" (FORMAT "parquet");
"""

con.execute(query)

con.close()

#### 5. Realizar a exclusão dos arquivos das amostras não unificadas

In [ ]:
%%bash
rm -f ../../data/1-sot/sample*

#### 6. Realizar a leitura da base unificada e padronizar o nome das colunas e dos tipos de dados

6.1. Além disso, filtram-se somente as viagens que aconteceram a partir de 01/01/2025.

In [ ]:
input_path = "../../data/1-sot/unified-tb-sor-fhvhv-trip-data.parquet"

sot = duckdb.sql(f"""
    SELECT 
        UPPER(TRIM(CAST(hvfhs_license_num AS VARCHAR))) AS identificador_empresa,
        UPPER(TRIM(CAST(dispatching_base_num AS VARCHAR))) AS identificador_base_despacho,
        UPPER(TRIM(CAST(originating_base_num AS VARCHAR))) AS identificador_base_origem,
        SUBSTRING(CAST(request_datetime AS VARCHAR), 1, 7) AS anomes_solicitacao,
        CAST(request_datetime AS DATE) AS data_solicitacao,
        CAST(request_datetime AS TIME) AS hora_solicitacao,
        CASE 
            WHEN STRFTIME(request_datetime, '%A') = 'Sunday' THEN 'a. Domingo' 
            WHEN STRFTIME(request_datetime, '%A') = 'Monday' THEN 'b. Segunda-feira' 
            WHEN STRFTIME(request_datetime, '%A') = 'Tuesday' THEN 'c. Terça-feira' 
            WHEN STRFTIME(request_datetime, '%A') = 'Wednesday' THEN 'd. Quarta-feira' 
            WHEN STRFTIME(request_datetime, '%A') = 'Thursday' THEN 'e. Quinta-feira' 
            WHEN STRFTIME(request_datetime, '%A') = 'Friday' THEN 'f. Sexta-feira' 
            WHEN STRFTIME(request_datetime, '%A') = 'Saturday' THEN 'g. Sábado' 
        END AS nome_dia_semana_solicitacao,
        SUBSTRING(CAST(on_scene_datetime AS VARCHAR), 1, 7) AS anomes_chegada_motorista,
        CAST(on_scene_datetime AS DATE) AS data_chegada_motorista,
        CAST(on_scene_datetime AS TIME) AS hora_chegada_motorista,
        CASE 
            WHEN STRFTIME(on_scene_datetime, '%A') = 'Sunday' THEN 'a. Domingo' 
            WHEN STRFTIME(on_scene_datetime, '%A') = 'Monday' THEN 'b. Segunda-feira' 
            WHEN STRFTIME(on_scene_datetime, '%A') = 'Tuesday' THEN 'c. Terça-feira' 
            WHEN STRFTIME(on_scene_datetime, '%A') = 'Wednesday' THEN 'd. Quarta-feira' 
            WHEN STRFTIME(on_scene_datetime, '%A') = 'Thursday' THEN 'e. Quinta-feira' 
            WHEN STRFTIME(on_scene_datetime, '%A') = 'Friday' THEN 'f. Sexta-feira' 
            WHEN STRFTIME(on_scene_datetime, '%A') = 'Saturday' THEN 'g. Sábado' 
        END AS nome_dia_semana_chegada,
        SUBSTRING(CAST(pickup_datetime AS VARCHAR), 1, 7) AS anomes_inicio_viagem,
        CAST(pickup_datetime AS DATE) AS data_inicio_viagem,
        CAST(pickup_datetime AS TIME) AS hora_inicio_viagem,
        CASE 
            WHEN STRFTIME(pickup_datetime, '%A') = 'Sunday' THEN 'a. Domingo' 
            WHEN STRFTIME(pickup_datetime, '%A') = 'Monday' THEN 'b. Segunda-feira' 
            WHEN STRFTIME(pickup_datetime, '%A') = 'Tuesday' THEN 'c. Terça-feira' 
            WHEN STRFTIME(pickup_datetime, '%A') = 'Wednesday' THEN 'd. Quarta-feira' 
            WHEN STRFTIME(pickup_datetime, '%A') = 'Thursday' THEN 'e. Quinta-feira' 
            WHEN STRFTIME(pickup_datetime, '%A') = 'Friday' THEN 'f. Sexta-feira' 
            WHEN STRFTIME(pickup_datetime, '%A') = 'Saturday' THEN 'g. Sábado' 
        END AS nome_dia_semana_inicio_viagem,
        DATE_DIFF('second', CAST(on_scene_datetime AS TIME), CAST(dropoff_datetime AS TIME)) AS diferenca_chegada_motorista_fim_viagem,
        SUBSTRING(CAST(dropoff_datetime AS VARCHAR), 1, 7) AS anomes_fim_viagem,
        CAST(dropoff_datetime AS DATE) AS data_fim_viagem,
        CAST(dropoff_datetime AS TIME) AS hora_fim_viagem,
        CASE 
            WHEN STRFTIME(dropoff_datetime, '%A') = 'Sunday' THEN 'a. Domingo' 
            WHEN STRFTIME(dropoff_datetime, '%A') = 'Monday' THEN 'b. Segunda-feira' 
            WHEN STRFTIME(dropoff_datetime, '%A') = 'Tuesday' THEN 'c. Terça-feira' 
            WHEN STRFTIME(dropoff_datetime, '%A') = 'Wednesday' THEN 'd. Quarta-feira' 
            WHEN STRFTIME(dropoff_datetime, '%A') = 'Thursday' THEN 'e. Quinta-feira' 
            WHEN STRFTIME(dropoff_datetime, '%A') = 'Friday' THEN 'f. Sexta-feira' 
            WHEN STRFTIME(dropoff_datetime, '%A') = 'Saturday' THEN 'g. Sábado' 
        END AS nome_dia_semana_fim_viagem,
        DATE_DIFF('second', CAST(pickup_datetime AS TIME), CAST(dropoff_datetime AS TIME)) AS diferenca_inicio_viagem_fim_viagem,
        CAST(PULocationID AS DECIMAL(16,2)) AS identificador_localizacao_embarque,
        CAST(DOLocationID AS DECIMAL(16,2)) AS identificador_localizacao_desembarque,
        CAST(trip_miles AS DECIMAL(16,2)) AS quantidade_milhas_viagem,
        CAST(trip_time AS DECIMAL(16,2)) AS quantidade_segundos_viagem,
        CAST(trip_miles AS DECIMAL(16,2)) / (CAST(trip_time AS DECIMAL(16,2)) / 3600) as valor_velocidade_media_viagem,
        CAST(base_passenger_fare AS DECIMAL(16,2)) AS valor_tarifa_base_passageiro,
        CAST(tolls AS DECIMAL(16,2)) AS valor_pedagios,
        CAST(bcf AS DECIMAL(16,2)) AS valor_fundo_black_car,
        CAST(sales_tax AS DECIMAL(16,2)) AS valor_imposto_vendas,
        CAST(congestion_surcharge AS DECIMAL(16,2)) AS valor_taxa_congestionamento,
        CAST(airport_fee AS DECIMAL(16,2)) AS valor_taxa_aeroporto,
        CAST(tips AS DECIMAL(16,2)) AS valor_gorjeta,
        CAST(driver_pay AS DECIMAL(16,2)) AS valor_pagamento_motorista,
        COALESCE(CAST(base_passenger_fare AS DECIMAL(16,2)), 0) + COALESCE(CAST(tolls AS DECIMAL(16,2)), 0) + COALESCE(CAST(bcf AS DECIMAL(16,2)), 0) + COALESCE(CAST(sales_tax AS DECIMAL(16,2)), 0) + COALESCE(CAST(congestion_surcharge AS DECIMAL(16,2)), 0) + COALESCE(CAST(airport_fee AS DECIMAL(16,2)), 0) + COALESCE(CAST(cbd_congestion_fee AS DECIMAL(16,2)), 0) + COALESCE(CAST(tips AS DECIMAL(16,2)), 0) AS valor_receita_total_viagem,
        CAST(shared_request_flag AS VARCHAR) AS indicador_corrida_compartilhada_solicitada,
        CAST(shared_match_flag AS VARCHAR) AS indicador_corrida_compartilhada_realizada,
        CAST(access_a_ride_flag AS VARCHAR) AS indicador_corrida_mta,
        CAST(wav_request_flag AS VARCHAR) AS indicador_veiculo_acessivel_solicitado,
        CAST(wav_match_flag AS VARCHAR) AS indicador_veiculo_acessivel_realizado,
        CAST(cbd_congestion_fee AS DECIMAL(16,2)) AS valor_taxa_congestionamento_cb
    FROM 
        read_parquet('{input_path}')
    WHERE
        SUBSTRING(CAST(request_datetime AS VARCHAR), 1, 7) >= '2025-01'
""").df()

sot.head(5)

#### 7. Realizar a análise exploratória dos dados

7.1. Principais métricas

In [ ]:
sot.describe()

7.2. Volume de viagens solicitadas por mês

In [ ]:
graph_1_travels_per_day = (
    sot
    .groupby(sot["anomes_solicitacao"])
    .size()
    .reset_index(name="quantidade_solicitacoes")
)

matplotlib.figure(figsize=(12, 4))

x = graph_1_travels_per_day["anomes_solicitacao"]
y = graph_1_travels_per_day["quantidade_solicitacoes"]

matplotlib.plot(x, y)

matplotlib.title("Quantidade de viagens solicitadas\npor ano e mês\n")

matplotlib.xticks(rotation=45)

matplotlib.gca().yaxis.set_visible(False)

for i, value in enumerate(y):
    matplotlib.text(x[i], value, f"{value:,}", ha='center', va='bottom')

matplotlib.tight_layout()

matplotlib.show()

7.3. Volume de viagens solicitadas por dia da semana

In [ ]:
graph_2_travels_per_weekday = (
    sot
    .groupby(sot["nome_dia_semana_solicitacao"])
    .size()
    .reset_index(name="quantidade_solicitacoes")
)

matplotlib.figure(figsize=(12, 4))

x = graph_2_travels_per_weekday["nome_dia_semana_solicitacao"]
y = graph_2_travels_per_weekday["quantidade_solicitacoes"]

matplotlib.plot(x, y)

matplotlib.title("Quantidade de viagens solicitadas\npor dia da semana\n")

matplotlib.xticks(rotation=45)

matplotlib.gca().yaxis.set_visible(False)

for i, value in enumerate(y):
    matplotlib.text(x[i], value, f"{value:,}", ha='center', va='bottom')

matplotlib.tight_layout()

matplotlib.show()

7.4. Velocidade média das viagens realizadas

In [ ]:
matplotlib.figure(figsize=(12, 4))

matplotlib.hist(sot["valor_velocidade_media_viagem"].dropna(), bins=100)

matplotlib.title("Distribuição da velocidade média\nde todas as corridas realizadas\n")

matplotlib.xticks(rotation=45)

matplotlib.gca().yaxis.set_visible(False)

matplotlib.tight_layout()

matplotlib.show()

7.5. Receita das corridas realizadas

In [ ]:
matplotlib.figure(figsize=(12, 4))

matplotlib.hist(sot["valor_receita_total_viagem"].dropna(), bins=100)

matplotlib.title("Distribuição da receita\nde todas as corridas realizadas\n")

matplotlib.xticks(rotation=45)

matplotlib.gca().yaxis.set_visible(False)

matplotlib.tight_layout()

matplotlib.show()

7.6. Relação entre as variáveis "distância" e "receita" das viagens

In [ ]:
matplotlib.figure(figsize=(12, 4))

matplotlib.scatter(sot["quantidade_milhas_viagem"], sot["valor_receita_total_viagem"])

matplotlib.title("Relação entre a distância (X) e a receita (Y)\ndas viagens realizadas\n")

matplotlib.xticks(rotation=45)

matplotlib.tight_layout()

matplotlib.show()

#### 8. Realizar a gravação do arquivo parquet da camada SOT

In [ ]:
con = duckdb.connect()

output_path = f"../../data/1-sot/tb-sot-fhvhv-trip-data-unified.parquet"

query = f"""
COPY 
(
    SELECT 
        *
    FROM 
        sot
)
TO "{output_path}" (FORMAT "parquet", COMPRESSION "snappy");
"""
    
con.execute(query)

con.close()

#### 9. Realizar a exclusão dos arquivos intermediários

In [ ]:
%%bash
rm -f ../../data/1-sot/unified*